# <center>Блок 5. Математика в ML. Часть I<center>
## <center>MATH&ML-6. Математический анализ в контексте задачи оптимизации. Часть III<center>
### <center>1.Введение<center>

In [1]:
from sympy import symbols, diff, solveset, Eq, init_printing
from sympy import *
import math

In [2]:
init_printing()

In [3]:
# Задание 1.4

x,y,w = symbols('x y w')
f = x**2 + 2*y**2 + w*(x + y - 20)
dx = Eq(f.diff(x),0)
dy = Eq(f.diff(y),0)
dw = Eq(f.diff(w),0)

sols = solve([dx,dy,dw],[x,y,w])
print(sols)

{w: -80/3, x: 40/3, y: 20/3}


### <center>2.Градиентный бустинг: примененние и модификации<center>
#### <center>Batch Gradient Descent<center>
#### <center>Stochastic Gradient Descent<center>
#### <center>Mini-Batch Gradient Descent<center>

In [4]:
# Задание 2.7

# Загружаем стандартный датасет об алмазах
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import linear_model
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import SGDRegressor
from sklearn.metrics import mean_squared_error

df = sns.load_dataset('diamonds')

# Удаляем часть признаков
df.drop(['depth', 'table', 'x', 'y', 'z'], axis=1, inplace=True)

# Закодируем категориальные признаки
df = pd.get_dummies(df, drop_first=True)

# Логарифмируем признаки
df['carat'] = np.log(1+df['carat'])
df['price'] = np.log(1+df['price'])

# Определяем целевую переменную и предикторы
X = df.drop(columns="price")
y = df["price"]

# Разделяем выборку на обучающую и тренировочную
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

# Обучаем модель
param_grid = {"loss": ["squared_error", "epsilon_insensitive"],
              "penalty": ["elasticnet"],
              "alpha": np.logspace(-3, 3, 10),
              "l1_ratio": np.linspace(0, 1, 10),
              "learning_rate": ["constant"],
              "eta0": np.logspace(-4, -1, 4)
}
grid_search = GridSearchCV(
    estimator=SGDRegressor(),
    param_grid=param_grid,
    n_jobs=-1
)

grid_search.fit(X_train,y_train)
y_test_predict = grid_search.predict(X_test)
print("Best model: {}".format(grid_search.best_estimator_))

Best model: SGDRegressor(alpha=0.001, l1_ratio=0.3333333333333333, learning_rate='constant',
             penalty='elasticnet')


In [5]:
model = SGDRegressor(
    loss="squared_error",
    penalty="elasticnet",
    alpha=0.001,
    l1_ratio=0.33333333,
    learning_rate='constant'
)
model.fit(X_train,y_train)
y_test_predict = model.predict(X_test)

mse = mean_squared_error(y_test,y_test_predict)
print(round(mse,3))

0.044


#### <center>Бонус: алгоритмы, основанные на градиентном спуске<center>
### <center>3.Метод Ньютона<center>

In [6]:
# Задание 3.1

from sympy import Symbol, simplify

x = Symbol("x")
f = 6*x**5 - 5*x**4 - 4*x**3 + 3*x**2
fx = f.diff(x)
x0 = 0.7
i = 1
while True:
    x_new = x0 - (f.subs(x,x0)/fx.subs(x,x0))
    if i > 3:
        break
    x0 = x_new
    i += 1
    print(round(x_new,3))

0.630
0.629
0.629


In [7]:
def func1(x):
    return 3*x**2 - 6*x -45

def func2(x):
    return 6*x -6

In [8]:
# Распишем алгоритм
initial_value = 42
iter_count = 0
x_curr = initial_value
epsilon = 0.0001 # точность
f = func1(x_curr)

while (abs(f)>epsilon):
    f = func1(x_curr)
    f_prime = func2(x_curr)
    x_curr = x_curr - (f)/(f_prime)
    iter_count += 1
    print(x_curr)

21.695121951219512
11.734125501243229
7.1123493600499685
5.365000391507974
5.015260627016227
5.000029000201801
5.000000000105126
5.000000000000001


In [9]:
# Можно обьединить все в одну функцию:
def newtons_method(f, fprime, x0, tol=0.0001):
    iter_count = 0
    x_curr = x0
    f_val = f(x_curr)
    while (abs(f_val)>tol):
        f_val = f(x_curr)
        f_prime_val = fprime(x_curr)
        x_curr = x_curr - (f_val)/(f_prime_val)
        iter_count += 1
    return x_curr

newtons_method(f=func1, fprime=func2, x0=50, tol=0.0001)

In [10]:
# Можно воспользоваться библиотекой scipy
from scipy.optimize import newton
newton(func=func1,fprime=func2,x0=50,tol=0.0001)

In [11]:
# Задание 3.6

def func1(x):
    return x**3 - 72*x - 220

def func2(x):
    return 3*x**2 - 72

initial_value = 12
iter_count = 0
x_curr = initial_value
epsilon = 0.0001 # точность
f = func1(x_curr)

while (abs(f)>epsilon):
    f = func1(x_curr)
    f_prime = func2(x_curr)
    x_curr = x_curr - (f)/(f_prime)
    iter_count += 1
    print(round(x_curr,3))

10.211
9.756
9.727
9.727
9.727


In [12]:
# Задание 3.7

def func1(x):
    return x**2 + 9*x - 5

def func2(x):
    return 2*x + 9

initial_value = 2.2
iter_count = 0
x_curr = initial_value
epsilon = 0.0001 # точность
f = func1(x_curr)

while (abs(f)>epsilon):
    f = func1(x_curr)
    f_prime = func2(x_curr)
    x_curr = x_curr - (f)/(f_prime)
    iter_count += 1
    print(round(x_curr,2))

0.73
0.53
0.52
0.52


In [13]:
# Задание 3.9

def func1(x):
    return 24*x**2 - 4*x

def func2(x):
    return 48*x - 4

initial_value = 42
iter_count = 0
x_curr = initial_value
epsilon = 0.0001 # точность
f = func1(x_curr)

while (abs(f)>epsilon):
    f = func1(x_curr)
    f_prime = func2(x_curr)
    x_curr = x_curr - (f)/(f_prime)
    iter_count += 1
    print(round(x_curr,3))

21.042
10.563
5.323
2.704
1.395
0.742
0.418
0.261
0.192
0.17
0.167
0.167
0.167


### <center>4.Квазиньютоновские методы<center>

In [14]:
# Подгрузим необходимые библиотеки
import numpy as np
from scipy.optimize import minimize
# Определим функцию. Вместо х и у можно взять координаты единого вектора
def func(x):
    return x[0]**2.0 + x[1]**2.0
# Теперь определим градиент для функции
def grad_func(x):
    return np.array([x[0]*2, x[1]*2])
# Зададим начальную точку
x_0 = [1.0,1.0]
# Определим алгоритм
result = minimize(func, x_0, method='BFGS', jac=grad_func)
# Выведем результаты
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s)= %.5f' % (solution, evaluation))

Статус оптимизации Optimization terminated successfully.
Количество оценок: 3
Решение: f([0. 0.])= 0.00000


In [15]:
# Повторим, используя вариацию L-BFGS-B
# Определим функцию
def func(x):
    return x[0]**2.0 + x[1]**2.0
# Определим градиент функции
def grad_func(x):
    return np.array([x[0]*2, x[1]*2])

# Определим начальную точку
x_0 = [1,1]
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
# Получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s)=%.5f' % (solution, evaluation))

Статус оптимизации CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Количество оценок: 3
Решение: f([0. 0.])=0.00000


In [17]:
# Задание 4.1
x,y = symbols('x y')
f = x**2 - x*y + y**2 + 9*x - 6*y + 20
print(f.diff(x))
print(f.diff(y))

def func(x):
    return x[0]**2.0 - x[0]*x[1] + x[1]**2.0 + 9.0*x[0] - 6.0*x[1] + 20
# Определим градиент функции
def grad_func(x):
    return np.array([2*x[0]-x[1]+9, -x[0]+2*x[1]-6])

# Определим начальную точку
x_0 = [-400,400]
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
# Получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s)=%.5f' % (solution, evaluation))

2*x - y + 9
-x + 2*y - 6
Статус оптимизации CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Количество оценок: 8
Решение: f([-4.  1.])=-1.00000


In [19]:
# Задание 4.4
x = Symbol('x')
f = x**2 - 3*x + 45
print(f.diff(x))

def func(x):
    return x**2.0 - 3.0*x + 45
# Определим градиент функции
def grad_func(x):
    return np.array(2*x - 3)

# Определим начальную точку
x_0 = 10
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='BFGS', jac=grad_func)
# Получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s)=%.5f' % (solution, evaluation))

2*x - 3
Статус оптимизации Optimization terminated successfully.
Количество оценок: 5
Решение: f([1.5])=42.75000


C:\Users\Smoking Shop\AppData\Local\Temp\ipykernel_23868\3699898031.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('Решение: f(%s)=%.5f' % (solution, evaluation))


In [20]:
# Задание 4.5
x = Symbol('x')
f = x**2 - 3*x + 45
print(f.diff(x))

def func(x):
    return x**2.0 - 3.0*x + 45
# Определим градиент функции
def grad_func(x):
    return np.array(2*x - 3)

# Определим начальную точку
x_0 = 10
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='L-BFGS-B', jac=grad_func)
# Получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s)=%.5f' % (solution, evaluation))

2*x - 3
Статус оптимизации CONVERGENCE: NORM_OF_PROJECTED_GRADIENT_<=_PGTOL
Количество оценок: 3
Решение: f([1.5])=42.75000


C:\Users\Smoking Shop\AppData\Local\Temp\ipykernel_23868\1203389864.py:21: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  print('Решение: f(%s)=%.5f' % (solution, evaluation))


In [24]:
# Задание 4.7
x,y = symbols('x y')
f = x**4 + 6*y**2 + 10
print(f.diff(x))
print(f.diff(y))

def func(x):
    return x[0]**4.0 + 6.0*x[1]**2.0 + 10
# Определим градиент функции
def grad_func(x):
    return np.array([4*x[0]**3, 12*x[1]])

# Определим начальную точку
x_0 = [100,100]
# реализуем алгоритм L-BFGS-B
result = minimize(func, x_0, method='BFGS', jac=grad_func)
# Получаем результат
print('Статус оптимизации %s' % result['message'])
print('Количество оценок: %d' % result['nfev'])
solution = result['x']
evaluation = func(solution)
print('Решение: f(%s)=%.5f' % (solution, evaluation))

4*x**3
12*y
Статус оптимизации Optimization terminated successfully.
Количество оценок: 37
Решение: f([1.31617159e-02 6.65344582e-14])=10.00000


### <center>5.Линейное программирование<center>

In [25]:
# Задача 5.5

def solve_sports_store():
    max_profit = -1
    best_x = 0
    best_y = 0
    
    # Перебираем все возможные целые значения в заданных диапазонах
    # x - количество футбольных мячей
    # y - количество бейсбольных мячей
    for x in range(35, 46):       # от 35 до 45 включительно
        for y in range(40, 56):   # от 40 до 55 включительно
            
            # Проверка ограничения на общее количество мячей
            if x + y <= 80:
                current_profit = 6 * x + 5.5 * y
                
                if current_profit > max_profit:
                    max_profit = current_profit
                    best_x = x
                    best_y = y

    return best_x, best_y, max_profit

# Запуск решения
football_balls, baseball_balls, total_profit = solve_sports_store()

print(f"Оптимальное количество футбольных мячей: {football_balls}")
print(f"Оптимальное количество бейсбольных мячей: {baseball_balls}")
print(f"Максимальная прибыль: ${total_profit}")

Оптимальное количество футбольных мячей: 40
Оптимальное количество бейсбольных мячей: 40
Максимальная прибыль: $460.0
